# Análise exploratória — anúncios de veículos usados (EUA)

Objetivo: entender a qualidade dos dados de `vehicles_us.csv` e quais fatores mais se relacionam com o preço dos anúncios, antes de construir o painel em Streamlit (`app.py`).

In [ ]:
import pandas as pd
import plotly.express as px

car_data = pd.read_csv('../vehicles_us.csv')
car_data.head()

## 1. Visão geral dos dados

In [ ]:
car_data.info()

In [ ]:
car_data.describe()

In [ ]:
# Proporção de valores ausentes por coluna
car_data.isna().mean().sort_values(ascending=False).round(3)

In [ ]:
car_data.duplicated().sum()

**Observações:**
- `is_4wd` tem ~50% de ausentes: a coluna só registra `1`, então ausente significa *não é 4x4*.
- `paint_color` (~18%), `odometer` (~15%), `cylinders` (~10%) e `model_year` (~7%) têm lacunas que podem ser preenchidas a partir de grupos semelhantes.
- Não há linhas duplicadas.

## 2. Tratamento de dados

In [ ]:
df = car_data.copy()

df['date_posted'] = pd.to_datetime(df['date_posted'])
df['manufacturer'] = df['model'].str.split().str[0]

df['is_4wd'] = df['is_4wd'].fillna(0).astype(bool)
df['paint_color'] = df['paint_color'].fillna('unknown')

# Ano e cilindros: mediana do mesmo modelo
for col in ['model_year', 'cylinders']:
    df[col] = df[col].fillna(df.groupby('model')[col].transform('median'))

# Quilometragem: mediana do mesmo ano de fabricação
df['odometer'] = df['odometer'].fillna(df.groupby('model_year')['odometer'].transform('median'))

df['model_year'] = df['model_year'].round().astype('Int64')
df['cylinders'] = df['cylinders'].round().astype('Int64')

df.isna().sum()

## 3. Distribuição de preços

In [ ]:
fig = px.histogram(df, x='price', nbins=100, title='Distribuição de preços (dados brutos)')
fig.show()

In [ ]:
print('Anúncios com preço abaixo de US$ 500:', (df['price'] < 500).sum())
df['price'].quantile([0.01, 0.05, 0.5, 0.95, 0.99])

A distribuição é fortemente assimétrica à direita, com valores extremos (até US$ 375 mil) e mais de mil anúncios abaixo de US$ 500 (vários com preço US$ 1, provavelmente placeholders). Para as análises seguintes, removemos preços abaixo de US$ 500 e acima do percentil 99, além da quilometragem acima do percentil 99.

In [ ]:
price_cap = df['price'].quantile(0.99)
odometer_cap = df['odometer'].quantile(0.99)
clean = df[df['price'].between(500, price_cap) & (df['odometer'] <= odometer_cap)]

print(f'{len(df)} -> {len(clean)} anúncios')
fig = px.histogram(clean, x='price', nbins=60, title='Distribuição de preços (sem outliers)')
fig.show()

## 4. Quilometragem e idade

In [ ]:
fig = px.histogram(clean, x='odometer', nbins=60, title='Distribuição da quilometragem')
fig.show()

In [ ]:
fig = px.histogram(clean, x='model_year', nbins=60, title='Ano do modelo dos veículos anunciados')
fig.show()

## 5. Relação entre preço e outras variáveis

In [ ]:
fig = px.scatter(clean, x='odometer', y='price', color='condition', opacity=0.4,
                 title='Preço × quilometragem por condição')
fig.show()

In [ ]:
fig = px.scatter(clean, x='model_year', y='price', opacity=0.3, title='Preço × ano do modelo')
fig.show()

In [ ]:
clean[['price', 'odometer', 'model_year', 'days_listed']].corr().round(2)

## 6. Preço por categoria

In [ ]:
order = ['new', 'like new', 'excellent', 'good', 'fair', 'salvage']
fig = px.box(clean, x='condition', y='price', category_orders={'condition': order},
             title='Preço por condição do veículo')
fig.show()

In [ ]:
median_by_type = clean.groupby('type')['price'].median().sort_values(ascending=False).reset_index()
fig = px.bar(median_by_type, x='type', y='price', title='Preço mediano por tipo de veículo')
fig.show()

In [ ]:
fig = px.histogram(clean, x='manufacturer', title='Anúncios por fabricante').update_xaxes(categoryorder='total descending')
fig.show()

In [ ]:
fig = px.histogram(clean, x='days_listed', nbins=60, title='Dias até o anúncio sair do ar')
fig.show()

## 7. Conclusões

- **Qualidade dos dados:** as lacunas foram preenchidas com medianas de grupos semelhantes (modelo ou ano), preservando todas as linhas; ~4% dos anúncios foram tratados como outliers de preço/quilometragem.
- **Preço:** mediana em torno de US$ 9 mil, com distribuição assimétrica; a maioria dos anúncios está entre ~US$ 2 mil e ~US$ 30 mil.
- **Idade e uso:** preço tem correlação moderada positiva com o ano do modelo (~0,5) e negativa com a quilometragem (~-0,5) — carros mais novos e menos rodados valem mais.
- **Condição:** o preço mediano cresce de forma consistente de *salvage/fair* (~US$ 2,5 mil) até *new* (~US$ 22 mil).
- **Tipo:** picapes e caminhões (*pickup*, *truck*) têm os maiores preços medianos; *hatchbacks* e sedãs, os menores.
- **Fabricantes:** Ford e Chevrolet concentram a maior parte dos anúncios.
- **Tempo de anúncio:** `days_listed` praticamente não se relaciona com o preço (correlação ≈ 0); a mediana é de cerca de 33 dias.

Esses achados orientam o painel: histogramas para as distribuições, dispersão para preço × quilometragem/ano e comparação de preço mediano por categoria.